# V-JEPA 2 perception — all-in-one demo

Single notebook, one Colab runtime. End-to-end:
1. Clone repo + install deps
2. Precompute V-JEPA 2 features on a small LIBERO Object slice
3. Train a tiny BC head on those features
4. Plot predicted vs ground-truth actions → `demo.png`

Total compute on T4: ~5 min with default settings (5 episodes, frame_stride=2).

**Runtime → Change runtime type → T4 GPU before running.**

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repo and install deps

In [ ]:
%cd /content
!git clone -b feat/vjepa2-perception https://github.com/PoCInnovation/LeWM-Robot.git || (cd LeWM-Robot && git fetch && git checkout feat/vjepa2-perception && git pull)
%cd /content/LeWM-Robot/vjepa-perception

In [ ]:
!pip install -q -r requirements.txt

## 3. Settings

Tweak here. Defaults give a decent demo without burning GPU time.

In [ ]:
MAX_EPISODES = 5      # number of source episodes to encode
FRAME_STRIDE = 2      # encode 1 frame out of N
BATCH_SIZE = 8        # encoder forward batch (lower if OOM on T4)
VAL_EPISODES = 1      # how many trailing episodes are held out for val
EPOCHS = 30

CACHE_DIR = '/content/cached_features/libero_object'
CKPT_DIR = '/content/checkpoints/bc_libero'
DEMO_PNG = '/content/demo.png'

## 4. Precompute V-JEPA 2 features

This is the slow part. Default config: ~3-5 min on T4.

In [ ]:
!python precompute_features.py \
    --src_repo lerobot/libero_object_image \
    --dst_dir "$CACHE_DIR" \
    --vjepa2_repo facebook/vjepa2-vitl-fpc64-256 \
    --batch_size $BATCH_SIZE \
    --dtype float16 \
    --device cuda \
    --num_workers 2 \
    --max_episodes $MAX_EPISODES \
    --frame_stride $FRAME_STRIDE

In [ ]:
import json, pathlib
meta = json.loads((pathlib.Path(CACHE_DIR) / 'metadata.json').read_text())
print(f"episodes: {meta['num_episodes']}  frames: {meta['num_frames']}  encoder_dim: {meta['encoder_dim']}")
print(f"per-episode lengths: {meta['episode_lengths']}")

## 5. Train the BC head

Mean-pool over 16×16 patches → MLP → action (~330k learnable params). The frozen ViT-L is *not* finetuned. Trains in <1 min.

In [ ]:
!python train.py \
    --cache_dir "$CACHE_DIR" \
    --output_dir "$CKPT_DIR" \
    --val_episodes $VAL_EPISODES \
    --epochs $EPOCHS \
    --batch_size 32 \
    --lr 1e-3

## 6. Demo plot: predicted vs ground-truth actions

In [ ]:
!python demo.py \
    --cache_dir "$CACHE_DIR" \
    --checkpoint "$CKPT_DIR/best.pt" \
    --output "$DEMO_PNG"

In [ ]:
from IPython.display import Image
Image(DEMO_PNG)

## 7. (Optional) Save artifacts to Google Drive

Run only if you want to keep `demo.png`, the checkpoint, and the cache after the runtime dies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/lewm_robot_demo
!cp -r "$CACHE_DIR" /content/drive/MyDrive/lewm_robot_demo/
!cp "$CKPT_DIR/best.pt" /content/drive/MyDrive/lewm_robot_demo/
!cp "$CKPT_DIR/history.json" /content/drive/MyDrive/lewm_robot_demo/
!cp "$DEMO_PNG" /content/drive/MyDrive/lewm_robot_demo/
!ls -la /content/drive/MyDrive/lewm_robot_demo/